In [ ]:
import os
import re
import random
from itertools import combinations

# Load documents

def load_doc(path):

    with open(path, 'r') as f:
        return f.read().strip()

doc_paths = {
    'D1': 'minhash/D1.txt',
    'D2': 'minhash/D2.txt',
    'D3': 'minhash/D3.txt',
    'D4': 'minhash/D4.txt',
}

docs = {name: load_doc(path) for name, path in doc_paths.items()}

# K-gram construction

def char_kgrams(text, k):
    # Character-level k-grams (set, no duplicates)
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def word_kgrams(text, k):
    # Word-level k-grams (set, no duplicates)
    words = text.split()
    return set(tuple(words[i:i+k]) for i in range(len(words) - k + 1))

print("K-GRAM COUNTS")
print("-" * 60)

for name, text in docs.items():

    cg2 = char_kgrams(text, 2)
    cg3 = char_kgrams(text, 3)
    wg2 = word_kgrams(text, 2)
    print(f"\n{name}:")
    print(f"  2-char grams : {len(cg2)}")
    print(f"  3-char grams : {len(cg3)}")
    print(f"  2-word grams : {len(wg2)}")

# Jaccard Similarity (exact)

def jaccard(set_a, set_b):

    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

print("\n")
print("QUESTION 1B: EXACT JACCARD SIMILARITIES (all pairs, all k-gram types)")
print("-" * 60)

doc_names = list(docs.keys())
pairs = list(combinations(doc_names, 2))

for gram_type, k_func, k in [('2-char', char_kgrams, 2), ('3-char', char_kgrams, 3), ('2-word', word_kgrams, 2)]:
    
    grams = {name: k_func(text, k) for name, text in docs.items()}
    print(f"\n--- {gram_type} grams ---")

    for a, b in pairs:
        j = jaccard(grams[a], grams[b])
        print(f"  J({a}, {b}) = {j:.6f}")

# Min-Hash

def get_universal_vocab(all_grams_list):

    vocab = set()
    for g in all_grams_list:
        vocab |= g
    return sorted(vocab)

def build_minhash_signature(gram_set, vocab, hash_funcs):
    # Build min-hash signature for a single document
    # Row indices where gram_set has a 1
    active_rows = [i for i, g in enumerate(vocab) if g in gram_set]
    sig = []

    for h in hash_funcs:
        if active_rows:
            sig.append(min(h(r) for r in active_rows))
        else:
            sig.append(float('inf'))

    return sig

def make_hash_funcs(t, num_rows, seed=42):
    #Generate t hash functions of form h(x) = (a*x + b) % p
    random.seed(seed)

    # Use a prime larger than num_rows
    p = 2**31 - 1  # Mersenne prime
    params = [(random.randint(1, p-1), random.randint(0, p-1)) for _ in range(t)]

    return [lambda x, a=a, b=b, p=p, n=num_rows: (a * x + b) % p for a, b in params]

def approx_jaccard(sig_a, sig_b):
    matches = sum(1 for a, b in zip(sig_a, sig_b) if a == b)
    return matches / len(sig_a)

print("\n")
print("QUESTION 1A: MIN-HASH APPROX JACCARD FOR D1 & D2 (3-char grams)")
print("-" * 60)

# Build 3-char gram sets for all docs (needed for universal vocab)
grams_3char = {name: char_kgrams(text, 3) for name, text in docs.items()}
vocab_3 = get_universal_vocab(list(grams_3char.values()))
num_rows = len(vocab_3)
print(f"Universal 3-char vocab size: {num_rows}")

t_values = [20, 60, 150, 300, 600]

print(f"\n{'t':>6}  {'Approx J(D1,D2)':>18}  {'Exact J(D1,D2)':>16}")
exact_j_d1d2 = jaccard(grams_3char['D1'], grams_3char['D2'])
print(f"{'exact':>6}  {'---':>18}  {exact_j_d1d2:>16.6f}")

for t in t_values:
    
    hash_funcs = make_hash_funcs(t, num_rows, seed=42)
    sig_d1 = build_minhash_signature(grams_3char['D1'], vocab_3, hash_funcs)
    sig_d2 = build_minhash_signature(grams_3char['D2'], vocab_3, hash_funcs)
    approx = approx_jaccard(sig_d1, sig_d2)
    print(f"  t={t:>4}  approx J = {approx:.6f}  (exact = {exact_j_d1d2:.6f})")


K-GRAM COUNTS
------------------------------------------------------------

D1:
  2-char grams : 263
  3-char grams : 765
  2-word grams : 279

D2:
  2-char grams : 262
  3-char grams : 762
  2-word grams : 278

D3:
  2-char grams : 269
  3-char grams : 828
  2-word grams : 337

D4:
  2-char grams : 255
  3-char grams : 698
  2-word grams : 232


QUESTION 1B: EXACT JACCARD SIMILARITIES (all pairs, all k-gram types)
------------------------------------------------------------

--- 2-char grams ---
  J(D1, D2) = 0.981132
  J(D1, D3) = 0.815700
  J(D1, D4) = 0.644444
  J(D2, D3) = 0.800000
  J(D2, D4) = 0.641270
  J(D3, D4) = 0.652997

--- 3-char grams ---
  J(D1, D2) = 0.977979
  J(D1, D3) = 0.580357
  J(D1, D4) = 0.305085
  J(D2, D3) = 0.568047
  J(D2, D4) = 0.305903
  J(D3, D4) = 0.312124

--- 2-word grams ---
  J(D1, D2) = 0.940767
  J(D1, D3) = 0.182342
  J(D1, D4) = 0.030242
  J(D2, D3) = 0.173664
  J(D2, D4) = 0.030303
  J(D3, D4) = 0.016071


QUESTION 1A: MIN-HASH APPROX JACCARD F